# Handling Important Missing Values: Milletvekili Şehir & Parti Look Up Table Creation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/tbmm_raw_tutanaklar/tutanaklar_28_konusmaci_bazli.csv'
df = pd.read_csv(file_path)
print(df.head())

  session_id  donem  yasama_yili  birlesim_no        date  \
0     28_1_1     28            1            1  2023-06-02   
1     28_1_1     28            1            1  2023-06-02   
2     28_1_1     28            1            1  2023-06-02   
3     28_1_1     28            1            1  2023-06-02   
4     28_1_1     28            1            1  2023-06-02   

                                       list_page_url  \
0  https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...   
1  https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...   
2  https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...   
3  https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...   
4  https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...   

                                        ham_page_url  \
0  https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...   
1  https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...   
2  https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...   
3  https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...   
4  https://www.t

### Data Types and Missing Values



In [ ]:
print('DataFrame Information:')
df.info()

print('\nMissing Values per Column:')
print(df.isnull().sum())

DataFrame Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199158 entries, 0 to 199157
Data columns (total 14 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   session_id        199158 non-null  object
 1   donem             199158 non-null  int64 
 2   yasama_yili       199158 non-null  int64 
 3   birlesim_no       199158 non-null  int64 
 4   date              199158 non-null  object
 5   list_page_url     199158 non-null  object
 6   ham_page_url      199158 non-null  object
 7   ozet_page_url     199158 non-null  object
 8   ham_html_url      199158 non-null  object
 9   konusmaci_raw     199158 non-null  object
 10  konusma_text      199158 non-null  object
 11  konusma_sirasi    199158 non-null  int64 
 12  konusma_uzunlugu  199158 non-null  int64 
 13  kelime_sayisi     199158 non-null  int64 
dtypes: int64(6), object(8)
memory usage: 21.3+ MB

Missing Values per Column:
session_id          0
donem        

### Analying Unique Speakers

Observing how many unique speakers are present in the `konusmaci_raw` column and how they look like.

In [ ]:
unique_speakers = df['konusmaci_raw'].unique()
unique_speakers_sorted = sorted(unique_speakers, key=lambda x: (len(x), x))
print("All unique speakers (sorted by length, then alphabetically):")
for speaker in unique_speakers_sorted:
    print(speaker)

All unique speakers (sorted by length, then alphabetically):
B
A.
AR
B.
ABD
AKP
BAĞ
DGD
HIT
II.
PAN
PKK
VII
DİSK
GİNE
III.
IŞİD
TRAB
TÜRK
MADEN
METAL
BİRTEK
DİSK GENEL
EĞİTİM SEN
BÜLENT KAYA
TÜRK EĞİTİM
SEMRA DİNÇER
ÖZGÜR KARABAT
GEÇİCİ MADDE I
EDİZ ÜN (Edirne)
SÜMEYYE BOZ(Muş)
BAŞKAN CELAL ADAN
EDİZ ÜN (Devamla)
EFKAN ALA (Bursa)
SÜMEYYE BOZ (Muş)
ZÜLKÜF UÇAR (Van)
ADEM ÇALKIN (Kars)
ALİ BOZAN (Mersin)
ALİ KARAOBA (UŞAK)
ALİ KARAOBA (Uşak)
ALİ YÜKSEL (Konya)
ALİ İNCİ (Devamla)
ALİ İNCİ (Sakarya)
ASU KAYA (Devamla)
AYŞE KEŞİR (Düzce)
HAKAN AKSU (Sivas)
KAAN KOÇ (Ardahan)
KAAN KOÇ (Devamla)
KADEM METE (Muğla)
KADİM METE (Muğla)
MAHMUT ÖZER (Ordu)
MERVAN GÜL (Siirt)
MURAT ÇAN (Samsun)
NEJLA DEMİR (Ağrı)
REFİK ÖZEN (Bursa)
RUKİYE TOY (Sivas)
SEYİT TORUN (ORDU)
SEYİT TORUN (Ordu)
SIRRI SAKİK (Ağrı)
TÜRKER ATEŞ (Bolu)
YILMAZ HUN (Iğdır)
ÖMER ÖZMEN (Aydın)
AHMET KILIÇ (Bursa)
AHMET ŞIK (Devamla)
ALİ BOZAN (Devamla)
ALİ TEMÜR (Devamla)
ALİ TEMÜR (Giresun)
ALİ ŞAHİN (Devamla)
ASU KAYA (Osmaniy

In [ ]:
import re
import numpy as np
import pandas as pd

# ============================================================
# SPEAKER CLEANING / ENRICHMENT FOR TBMM SPEECH DATA
# Assumes your main dataframe is called: df
# Assumes raw speaker column is called: konusmaci_raw
# ============================================================

RAW_COL = "konusmaci_raw"

# ------------------------------------------------------------
# OPTIONAL: If you later have external milletvekili-party info,
# set PARTY_LOOKUP_PATH to a CSV/XLSX file.
#
# Expected possible columns:
#   name:  "konusmaci_clean", "ad_soyad", "ad soyad", "isim", "name", "milletvekili"
#   party: "parti", "party", "siyasi_parti"
#   city:  "sehir", "şehir", "il"
#
# Example:
# PARTY_LOOKUP_PATH = "/content/drive/MyDrive/milletvekili_party_info.xlsx"
# ------------------------------------------------------------

PARTY_LOOKUP_PATH = None


# ============================================================
# BASIC NORMALIZATION HELPERS
# ============================================================

TR_UPPER_TRANS = str.maketrans({
    "i": "İ",
    "ı": "I",
})

def norm_space(x):
    if pd.isna(x):
        return ""
    x = str(x).replace("\xa0", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def tr_upper(x):
    return norm_space(x).translate(TR_UPPER_TRANS).upper()

def clean_key(x):
    """
    Strong comparison key for joining names.
    Keeps Turkish letters, removes punctuation and extra spaces.
    """
    x = tr_upper(x)
    x = re.sub(r"[^\wÇĞİÖŞÜÂÎÛçğıöşüâîû ]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


# ============================================================
# CITY EXTRACTION
# ============================================================

PROVINCES = [
    "Adana", "Adıyaman", "Afyonkarahisar", "Ağrı", "Amasya", "Ankara",
    "Antalya", "Artvin", "Aydın", "Balıkesir", "Bilecik", "Bingöl",
    "Bitlis", "Bolu", "Burdur", "Bursa", "Çanakkale", "Çankırı",
    "Çorum", "Denizli", "Diyarbakır", "Edirne", "Elazığ", "Erzincan",
    "Erzurum", "Eskişehir", "Gaziantep", "Giresun", "Gümüşhane",
    "Hakkâri", "Hatay", "Isparta", "Mersin", "İstanbul", "İzmir",
    "Kars", "Kastamonu", "Kayseri", "Kırklareli", "Kırşehir",
    "Kocaeli", "Konya", "Kütahya", "Malatya", "Manisa",
    "Kahramanmaraş", "Mardin", "Muğla", "Muş", "Nevşehir",
    "Niğde", "Ordu", "Rize", "Sakarya", "Samsun", "Siirt",
    "Sinop", "Sivas", "Tekirdağ", "Tokat", "Trabzon", "Tunceli",
    "Şanlıurfa", "Uşak", "Van", "Yozgat", "Zonguldak", "Aksaray",
    "Bayburt", "Karaman", "Kırıkkale", "Batman", "Şırnak",
    "Bartın", "Ardahan", "Iğdır", "Yalova", "Karabük", "Kilis",
    "Osmaniye", "Düzce"
]

CITY_MAP = {tr_upper(c): c for c in PROVINCES}

CITY_ALIASES = {
    "ELÂZIĞ": "Elazığ",
    "ELAZIĞ": "Elazığ",
    "HAKKARI": "Hakkâri",
    "HAKKÂRİ": "Hakkâri",
    "AFYON": "Afyonkarahisar",
    "AFYONKARAHİSAR": "Afyonkarahisar",
    "UŞAK": "Uşak",
    "ORDU": "Ordu",
    "MUŞ": "Muş",
    "İSTANBUL": "İstanbul",
    "ISTANBUL": "İstanbul",
    "İZMİR": "İzmir",
    "IZMIR": "İzmir",
    "ŞANLIURFA": "Şanlıurfa",
    "SANLIURFA": "Şanlıurfa",
    "DİYARBAKIR": "Diyarbakır",
    "DIYARBAKIR": "Diyarbakır",
    "KIRIKKALE": "Kırıkkale",
    "KIRKLARELİ": "Kırklareli",
    "KIRKLARELI": "Kırklareli",
    "KIRŞEHİR": "Kırşehir",
    "KIRSEHIR": "Kırşehir",
}

DEVAM_WORDS = {
    "DEVAM",
    "DEVAMLA",
    "DEVAMEN",
    "DEVAM EDEREK",
}

def canonical_city(x):
    x = norm_space(x).strip(" .")
    up = tr_upper(x)

    if up in DEVAM_WORDS:
        return pd.NA

    if up in CITY_ALIASES:
        return CITY_ALIASES[up]

    if up in CITY_MAP:
        return CITY_MAP[up]

    # If it is not recognized but is inside parentheses, keep it as-is.
    # This helps you inspect typos such as "Terkirdağ" later.
    return x if x else pd.NA

def extract_city_from_raw(raw):
    s = norm_space(raw)

    # Finds things like (Ankara), (Devamla), ( Muş)
    parens = re.findall(r"\(([^()]*)\)", s)

    for p in reversed(parens):
        p_clean = norm_space(p).strip(" .")
        p_up = tr_upper(p_clean)

        if not p_clean:
            continue

        if p_up in DEVAM_WORDS:
            continue

        return canonical_city(p_clean)

    return pd.NA


# ============================================================
# BAD SPEAKER LABELS
# These are not real speaker identities.
# I intentionally do NOT include real-looking names like
# BÜLENT KAYA, SEMRA DİNÇER, ÖZGÜR KARABAT here,
# because they may be valid speakers without city info.
# ============================================================

BAD_EXACT_LABELS = {
    "B",
    "A.",
    "AR",
    "B.",
    "ABD",
    "AKP",
    "BAĞ",
    "DGD",
    "HIT",
    "II.",
    "III.",
    "VII",
    "PAN",
    "PKK",
    "DİSK",
    "GİNE",
    "IŞİD",
    "TRAB",
    "TÜRK",
    "MADEN",
    "METAL",
    "BİRTEK",
    "DİSK GENEL",
    "EĞİTİM SEN",
    "TÜRK EĞİTİM",
    "GEÇİCİ MADDE I",
    "İSTANBUL ÜNİVERSİTESİ",
    "TÜRKİYE CUMHURİYETİ HÜKÜMETİ İLE BOSNA",
    "TÜRKİYE CUMHURİYETİ HÜKÜMETİ İLE TÜRKMENİSTAN HÜKÜMETİ ARASINDA ORTAK TÜRKMEN",
}

BAD_EXACT_UPPER = {tr_upper(x) for x in BAD_EXACT_LABELS}

def looks_like_bad_label(raw):
    up = tr_upper(raw)

    if not up:
        return True

    if up in BAD_EXACT_UPPER:
        return True

    # Obvious heading / agreement / institution fragments
    if re.search(r"HÜKÜMETİ İLE|ÜNİVERSİTESİ|GEÇİCİ MADDE", up):
        return True

    # Very short all-caps fragments without parentheses are usually parse errors
    compact = re.sub(r"[\s.]+", "", up)
    if len(compact) <= 4 and "(" not in up:
        if not up.startswith(("BAŞKAN", "TBMM", "CUMHURBAŞKANI")):
            return True

    return False


# ============================================================
# PARTY NORMALIZATION
# ============================================================

PARTY_PATTERNS = [
    (r"\bADALET VE KALKINMA PART[İI]S[İI]\b|\bAK\s*PART[İI]\b", "AK PARTİ"),
    (r"\bCUMHUR[İI]YET HALK PART[İI]S[İI]\b|\bCHP\b", "CHP"),
    (r"\bM[İI]LL[İI]YETÇ[İI] HAREKET PART[İI]S[İI]\b|\bMHP\b", "MHP"),
    (r"\b[İI]Y[İI]\s+PART[İI]\b", "İYİ PARTİ"),
    (r"\bDEMOKRAT[İI]K SOL PART[İI]\b", "DSP"),
    (r"\bDEMOKRAS[İI] VE ATILIM PART[İI]S[İI]\b", "DEVA PARTİSİ"),
    (r"\bDEMOKRAT PART[İI]\b", "DEMOKRAT PARTİ"),
    (r"\bHALKLARIN EŞ[İI]TL[İI]K VE DEMOKRAS[İI] PART[İI]S[İI]\b|\bHEDEP\b", "HEDEP / DEM PARTİ"),
    (r"\bDEM\s+PART[İI]?\b", "DEM PARTİ"),
    (r"\bYEŞ[İI]L SOL PART[İI](?:S[İI])?\b", "YEŞİL SOL PARTİ"),
    (r"\bSAADET PART[İI]S[İI]\b", "SAADET PARTİSİ"),
    (r"\bYEN[İI]\s+YOL(?:\s+PART[İI]S[İI])?\b", "YENİ YOL PARTİSİ"),
    (r"\bTÜRK[İI]YE [İI]ŞÇ[İI] PART[İI]S[İI]\b", "TİP"),
    (r"\bHÜR DAVA PART[İI]S[İI]\b", "HÜDA PAR"),
    (r"\bYEN[İI]DEN REFAH PART[İI]S[İI]\b", "YENİDEN REFAH PARTİSİ"),
    (r"\bEMEK PART[İI]S[İI]\b", "EMEP"),
    (r"\bGELECEK PART[İI]S[İI]\b", "GELECEK PARTİSİ"),
]

def normalize_party_from_text(raw):
    up = tr_upper(raw)

    for pattern, party in PARTY_PATTERNS:
        if re.search(pattern, up):
            return party

    return pd.NA


# ============================================================
# GROUP PREFIX DETECTION
# Examples:
#   CHP GRUBU ADINA EDİZ ÜN
#   AK PARTİ GRUBU ADINA EFKAN ALA
#   DEM PARTİ GRUBUNA BURCUGÜL ÇUBUK
#   İYİ PARTİ GBURU ADINA TURHAN ÇÖMEZ
# ============================================================

GROUP_PREFIX_PATTERN = (
    r"^(?:"
    r"AK\s*PART[İI]|"
    r"CHP|"
    r"DEM\s+PART[İI]?|"
    r"DEM\s+PART|"
    r"HEDEP|"
    r"YEŞ[İI]L\s+SOL\s+PART[İI](?:S[İI])?|"
    r"MHP|"
    r"[İI]Y[İI]\s+PART[İI]|"
    r"SAADET\s+PART[İI]S[İI]|"
    r"YEN[İI]\s+YOL(?:\s+PART[İI]S[İI])?"
    r")\s+"
    r"(?:MECL[İI]S\s+)?"
    r"(?:GRUBU|GRUBA|GBURU|GBUBU|GURUBU|GRUP|GRUBUNA)"
    r"\s*(?:ADINA)?\s+"
)

def is_group_adina(raw):
    up = tr_upper(raw)
    return bool(re.search(
        r"(GRUBU|GRUBA|GBURU|GBUBU|GURUBU|GRUBUNA|GRUP)\s*(ADINA)?",
        up
    ))


# ============================================================
# SPEAKER TYPE CLASSIFICATION
# ============================================================

def classify_speaker_type(raw):
    up = tr_upper(raw)

    if looks_like_bad_label(raw):
        return "parse_error"

    if up.startswith("TBMM BAŞKANI"):
        return "tbmm başkanı"

    if up.startswith("BAŞKAN "):
        return "başkan"

    if up.startswith("CUMHURBAŞKANI YARDIMCISI"):
        return "cumhurbaşkanı yardımcısı"

    if up.startswith("CUMHURBAŞKANI "):
        return "cumhurbaşkanı"

    if up.startswith("FİLİSTİN DEVLET BAŞKANI") or " DEVLET BAŞKANI " in up:
        return "devlet başkanı"

    if "BAŞDENETÇİSİ" in up:
        return "kamu başdenetçisi"

    if re.search(r"\bBAKANI\b", up):
        return "bakan"

    if re.search(r"KOM[İI]SYON\w*\s+BAŞKAN\s*VEK[İI]L[İI]", up):
        return "komisyon başkan vekili"

    if re.search(r"KOM[İI]SYON\w*\s+BAŞKAN(?:I)?\b", up):
        return "komisyon başkanı"

    if re.search(r"KOM[İI]SYON\w*\s+SÖZCÜSÜ|KOM[İI]SYON\w*\s+SÖZÜSÜ", up):
        return "komisyon sözcüsü"

    if re.search(r"GENEL BAŞKANI\s+ADINA", up):
        return "parti genel başkanı adına"

    if re.search(r"EŞ\s+GENEL BAŞKANI|GENEL BAŞKANI", up):
        return "parti genel başkanı"

    if re.search(r"GRUP BAŞKAN VEK[İI]L[İI]", up):
        return "grup başkan vekili"

    if re.search(r"MECL[İI]S GRUBU BAŞKANI|GRUBU BAŞKANI", up):
        return "meclis grubu başkanı"

    if is_group_adina(raw):
        return "milletvekili (grup adına)"

    return "milletvekili"


# ============================================================
# CLEAN NAME EXTRACTION
# ============================================================

def strip_prefix_using_upper(original, pattern):
    """
    Applies regex to Turkish-uppercase version but slices original string.
    This avoids Turkish case issues while preserving original characters.
    """
    original = norm_space(original)
    up = tr_upper(original)

    m = re.match(pattern, up)
    if m:
        return norm_space(original[m.end():])

    return original

PREFIX_PATTERNS_TO_REMOVE = [
    # Titles / institutional roles
    r"^SAYIN\s+",
    r"^TBMM\s+BAŞKANI\s+",
    r"^BAŞKAN\s+",
    r"^CUMHURBAŞKANI\s+YARDIMCISI\s+",
    r"^CUMHURBAŞKANI\s+",
    r"^F[İI]L[İI]ST[İI]N\s+DEVLET\s+BAŞKANI\s+",
    r"^KAMU\s+BAŞDENETÇ[İI]S[İI]\s+",

    # Commission roles
    r"^.*?\bKOM[İI]SYON\w*\s+(?:BAŞKAN\s*VEK[İI]L[İI]|BAŞKANVEK[İI]L[İI]|BAŞKAN(?:I)?|SÖZCÜSÜ|SÖZÜSÜ)\s+",

    # Ministers
    r"^.*?\bBAKANI\s+",

    # Party leader / group leader forms
    r"^.*?\b(?:EŞ\s+)?GENEL\s+BAŞKANI(?:\s+VE\s+MECL[İI]S\s+GRUBU\s+BAŞKANI)?(?:\s+ADINA)?\s+",
    r"^.*?\b(?:MECL[İI]S\s+)?GRUBU\s+BAŞKANI\s+",
    r"^.*?\b(?:MECL[İI]S\s+)?GRUP\s+BAŞKAN\s+VEK[İI]L[İI]\s+",
]

def clean_speaker_name(raw):
    if looks_like_bad_label(raw):
        return pd.NA

    s = norm_space(raw)

    # Remove all parenthetical parts: (Ankara), (Devamla), etc.
    s = re.sub(r"\([^()]*\)", " ", s)
    s = norm_space(s)

    # Remove group-speech prefix first:
    # CHP GRUBU ADINA X -> X
    s = strip_prefix_using_upper(s, GROUP_PREFIX_PATTERN)

    # Remove role/title prefixes
    for pattern in PREFIX_PATTERNS_TO_REMOVE:
        new_s = strip_prefix_using_upper(s, pattern)
        if new_s != s:
            s = new_s
            break

    # Sometimes after one removal another prefix remains
    # Example: typo-heavy group/title combinations
    for pattern in PREFIX_PATTERNS_TO_REMOVE:
        new_s = strip_prefix_using_upper(s, pattern)
        if new_s != s:
            s = new_s

    # Final cleanup
    s = re.sub(r"\bADINA\b", " ", s, flags=re.IGNORECASE)
    s = re.sub(r"[.:;,\-–—]+$", "", s)
    s = re.sub(r"\s+", " ", s).strip()

    if not s:
        return pd.NA

    # Return as normalized uppercase for consistent grouping
    return tr_upper(s)


# ============================================================
# MAIN ENRICHMENT
# ============================================================

df_speaker_enriched = df.copy()

# Keep raw as-is; create new columns
df_speaker_enriched["konusmaci_clean"] = df_speaker_enriched[RAW_COL].apply(clean_speaker_name)
df_speaker_enriched["konusmaci_sehir"] = df_speaker_enriched[RAW_COL].apply(extract_city_from_raw)
df_speaker_enriched["konusmaci_tip"] = df_speaker_enriched[RAW_COL].apply(classify_speaker_type)
df_speaker_enriched["konusmaci_grup_adina"] = df_speaker_enriched[RAW_COL].apply(is_group_adina)
df_speaker_enriched["konusmaci_parti"] = df_speaker_enriched[RAW_COL].apply(normalize_party_from_text)

# Parse-error flag
df_speaker_enriched["speaker_parse_error"] = (
    df_speaker_enriched[RAW_COL].apply(looks_like_bad_label)
    | df_speaker_enriched["konusmaci_clean"].isna()
)

# For parse errors, keep raw but blank analytical speaker fields
parse_error_cols = [
    "konusmaci_clean",
    "konusmaci_sehir",
    "konusmaci_parti",
]
df_speaker_enriched.loc[df_speaker_enriched["speaker_parse_error"], parse_error_cols] = pd.NA
df_speaker_enriched.loc[df_speaker_enriched["speaker_parse_error"], "konusmaci_tip"] = "parse_error"


# ============================================================
# FILL CITY FOR (Devamla) ROWS USING SAME CLEAN NAME
# Example:
#   ALİ KARAOBA (Uşak)
#   ALİ KARAOBA (Devamla)
# The second row gets Uşak.
# ============================================================

NO_CITY_ROLES = {
    "başkan",
    "tbmm başkanı",
    "cumhurbaşkanı",
    "cumhurbaşkanı yardımcısı",
    "bakan",
    "devlet başkanı",
    "kamu başdenetçisi",
    "parse_error",
}

def mode_or_na(series):
    s = series.dropna()
    if s.empty:
        return pd.NA
    return s.mode().iloc[0]

name_to_city = (
    df_speaker_enriched
    .loc[
        df_speaker_enriched["konusmaci_clean"].notna()
        & df_speaker_enriched["konusmaci_sehir"].notna(),
        ["konusmaci_clean", "konusmaci_sehir"]
    ]
    .groupby("konusmaci_clean")["konusmaci_sehir"]
    .agg(mode_or_na)
)

city_fill_mask = (
    df_speaker_enriched["konusmaci_sehir"].isna()
    & df_speaker_enriched["konusmaci_clean"].notna()
    & ~df_speaker_enriched["konusmaci_tip"].isin(NO_CITY_ROLES)
)

df_speaker_enriched.loc[city_fill_mask, "konusmaci_sehir"] = (
    df_speaker_enriched.loc[city_fill_mask, "konusmaci_clean"].map(name_to_city)
)




def find_column(columns, candidates):
    lower_map = {str(c).strip().lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

if PARTY_LOOKUP_PATH:
    if PARTY_LOOKUP_PATH.lower().endswith((".xlsx", ".xls")):
        party_lookup_df = pd.read_excel(PARTY_LOOKUP_PATH)
    else:
        party_lookup_df = pd.read_csv(PARTY_LOOKUP_PATH)

    name_col = find_column(
        party_lookup_df.columns,
        [
            "konusmaci_clean",
            "ad_soyad",
            "ad soyad",
            "isim",
            "name",
            "milletvekili",
            "milletvekili_ad_soyad",
        ]
    )

    party_col = find_column(
        party_lookup_df.columns,
        [
            "parti",
            "party",
            "siyasi_parti",
            "siyasi parti",
        ]
    )

    city_col = find_column(
        party_lookup_df.columns,
        [
            "sehir",
            "şehir",
            "il",
        ]
    )

    if name_col is None:
        raise ValueError(
            "External party file must contain a name column such as "
            "'ad_soyad', 'konusmaci_clean', 'isim', 'name', or 'milletvekili'."
        )

    lookup = party_lookup_df.copy()
    lookup["_name_key"] = lookup[name_col].apply(clean_key)

    keep_cols = ["_name_key"]

    if party_col is not None:
        lookup["_lookup_party"] = lookup[party_col].apply(normalize_party_from_text)
        # If normalize_party_from_text cannot recognize it, keep original party text.
        lookup["_lookup_party"] = lookup["_lookup_party"].fillna(lookup[party_col].apply(norm_space))
        keep_cols.append("_lookup_party")

    if city_col is not None:
        lookup["_lookup_city"] = lookup[city_col].apply(canonical_city)
        keep_cols.append("_lookup_city")

    lookup = lookup[keep_cols].drop_duplicates("_name_key", keep="first")

    df_speaker_enriched["_name_key"] = df_speaker_enriched["konusmaci_clean"].apply(clean_key)

    df_speaker_enriched = df_speaker_enriched.merge(
        lookup,
        on="_name_key",
        how="left"
    )

    NO_PARTY_ROLES = {
        "başkan",
        "tbmm başkanı",
        "cumhurbaşkanı",
        "cumhurbaşkanı yardımcısı",
        "bakan",
        "devlet başkanı",
        "kamu başdenetçisi",
        "parse_error",
    }

    if "_lookup_party" in df_speaker_enriched.columns:
        party_fill_mask = (
            df_speaker_enriched["konusmaci_parti"].isna()
            & df_speaker_enriched["_lookup_party"].notna()
            & ~df_speaker_enriched["konusmaci_tip"].isin(NO_PARTY_ROLES)
        )

        df_speaker_enriched.loc[party_fill_mask, "konusmaci_parti"] = (
            df_speaker_enriched.loc[party_fill_mask, "_lookup_party"]
        )

    if "_lookup_city" in df_speaker_enriched.columns:
        city_fill_mask_2 = (
            df_speaker_enriched["konusmaci_sehir"].isna()
            & df_speaker_enriched["_lookup_city"].notna()
            & ~df_speaker_enriched["konusmaci_tip"].isin(NO_CITY_ROLES)
        )

        df_speaker_enriched.loc[city_fill_mask_2, "konusmaci_sehir"] = (
            df_speaker_enriched.loc[city_fill_mask_2, "_lookup_city"]
        )

    drop_cols = [
        c for c in ["_name_key", "_lookup_party", "_lookup_city"]
        if c in df_speaker_enriched.columns
    ]
    df_speaker_enriched = df_speaker_enriched.drop(columns=drop_cols)


# ============================================================
# FINAL CLEANED DATAFRAMES
# ============================================================

# Main enriched dataframe: keeps everything, including parse-error rows
df_speaker_enriched = df_speaker_enriched.copy()

# Analysis-ready dataframe: excludes bad speaker-label rows
df_speaker_clean = df_speaker_enriched[
    ~df_speaker_enriched["speaker_parse_error"]
].copy()

# Suspicious rows saved separately for transparency
df_speaker_parse_errors = df_speaker_enriched[
    df_speaker_enriched["speaker_parse_error"]
].copy()


# ============================================================
# QUICK CHECKS
# ============================================================

print("Original rows:", len(df))
print("Enriched rows:", len(df_speaker_enriched))
print("Clean speaker rows:", len(df_speaker_clean))
print("Speaker parse-error rows:", len(df_speaker_parse_errors))

print("\nSpeaker type counts:")
print(df_speaker_enriched["konusmaci_tip"].value_counts(dropna=False))

print("\nParty counts:")
print(df_speaker_enriched["konusmaci_parti"].value_counts(dropna=False).head(30))

print("\nSample cleaned speakers:")
display(
    df_speaker_enriched[
        [
            RAW_COL,
            "konusmaci_clean",
            "konusmaci_sehir",
            "konusmaci_tip",
            "konusmaci_parti",
            "konusmaci_grup_adina",
            "speaker_parse_error",
        ]
    ]
    .drop_duplicates()
    .head(80)
)

print("\nSample parse errors:")
display(
    df_speaker_parse_errors[
        [
            RAW_COL,
            "konusmaci_clean",
            "konusmaci_sehir",
            "konusmaci_tip",
            "konusmaci_parti",
            "speaker_parse_error",
        ]
    ]
    .drop_duplicates()
    .head(80)
)


# ============================================================
# SAVE
# ============================================================

df_speaker_enriched.to_csv("tbmm_speeches_speaker_enriched.csv", index=False)
df_speaker_clean.to_csv("tbmm_speeches_speaker_clean.csv", index=False)
df_speaker_parse_errors.to_csv("tbmm_speaker_parse_errors.csv", index=False)

Original rows: 199158
Enriched rows: 199158
Clean speaker rows: 198951
Speaker parse-error rows: 207

Speaker type counts:
konusmaci_tip
milletvekili                 115912
başkan                        70740
milletvekili (grup adına)      7506
bakan                          2552
komisyon sözcüsü               1091
cumhurbaşkanı yardımcısı        511
komisyon başkanı                473
parse_error                     207
komisyon başkan vekili           63
parti genel başkanı              48
parti genel başkanı adına        18
meclis grubu başkanı             14
cumhurbaşkanı                    11
grup başkan vekili                9
devlet başkanı                    1
kamu başdenetçisi                 1
tbmm başkanı                      1
Name: count, dtype: int64

Party counts:
konusmaci_parti
<NA>                     191563
CHP                        1538
AK PARTİ                   1349
İYİ PARTİ                  1345
DEM PARTİ                  1271
SAADET PARTİSİ              733
YE

,konusmaci_raw,konusmaci_clean,konusmaci_sehir,konusmaci_tip,konusmaci_parti,konusmaci_grup_adina,speaker_parse_error
0,BAŞKAN DEVLET BAHÇELİ,DEVLET BAHÇELİ,<NA>,başkan,<NA>,False,False
10,OSMAN GÖKÇEK (Ankara),OSMAN GÖKÇEK,Ankara,milletvekili,<NA>,False,False
11,MURAT EMİR (Ankara),MURAT EMİR,Ankara,milletvekili,<NA>,False,False
12,OSMAN GÖKÇEK (Devamla),OSMAN GÖKÇEK,Ankara,milletvekili,<NA>,False,False
21,ŞAHİN TİN (Denizli),ŞAHİN TİN,Denizli,milletvekili,<NA>,False,False
...,...,...,...,...,...,...,...
584,AYHAN BARUT (Adana),AYHAN BARUT,Adana,milletvekili,<NA>,False,False
586,MUSTAFA ADIGÜZEL (Ordu),MUSTAFA ADIGÜZEL,Ordu,milletvekili,<NA>,False,False
588,HASAN ÖZTÜRKMEN (Gaziantep),HASAN ÖZTÜRKMEN,Gaziantep,milletvekili,<NA>,False,False
590,ORHAN SÜMER (Adana),ORHAN SÜMER,Adana,milletvekili,<NA>,False,False



Sample parse errors:


,konusmaci_raw,konusmaci_clean,konusmaci_sehir,konusmaci_tip,konusmaci_parti,speaker_parse_error
4896,İSTANBUL ÜNİVERSİTESİ,<NA>,<NA>,parse_error,<NA>,True
5021,AKP,<NA>,<NA>,parse_error,<NA>,True
10690,GİNE,<NA>,<NA>,parse_error,<NA>,True
18459,TÜRK EĞİTİM,<NA>,<NA>,parse_error,<NA>,True
21478,TÜRK,<NA>,<NA>,parse_error,<NA>,True
23514,AR,<NA>,<NA>,parse_error,<NA>,True
30536,TRAB,<NA>,<NA>,parse_error,<NA>,True
33812,B,<NA>,<NA>,parse_error,<NA>,True
42305,VII,<NA>,<NA>,parse_error,<NA>,True
57690,BAĞ,<NA>,<NA>,parse_error,<NA>,True


#Adding MP Party Information

In [ ]:
import re
import os
import requests
import pandas as pd
from google.colab import drive

# ============================================================
# PATH SETTINGS
# ============================================================

drive.mount("/content/drive")

OUT_DIR = "/content/drive/MyDrive/tbmm_raw_tutanaklar"
os.makedirs(OUT_DIR, exist_ok=True)

URL = "https://tbmm.gov.tr/milletvekili/eposta-liste"

CSV_OUT = f"{OUT_DIR}/milletvekili_party_city_lookup.csv"
PARQUET_OUT = f"{OUT_DIR}/milletvekili_party_city_lookup.parquet"


# ============================================================
# NORMALIZATION HELPERS
# ============================================================

TR_UPPER_TRANS = str.maketrans({
    "i": "İ",
    "ı": "I",
})

def norm_space(x):
    if pd.isna(x):
        return ""
    x = str(x).replace("\xa0", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def tr_upper(x):
    return norm_space(x).translate(TR_UPPER_TRANS).upper()

def clean_key(x):
    x = tr_upper(x)
    x = re.sub(r"[^\wÇĞİÖŞÜÂÎÛ ]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


# ============================================================
# PARTY NORMALIZATION
# ============================================================

def normalize_party(x):
    x_raw = norm_space(x)
    x_up = tr_upper(x_raw)

    party_map = {
        "AK PARTİ": "AK PARTİ",
        "AK PARTI": "AK PARTİ",
        "AK PARTİSİ": "AK PARTİ",
        "CHP": "CHP",
        "MHP": "MHP",
        "DEM PARTİ": "DEM PARTİ",
        "DEM PARTI": "DEM PARTİ",
        "İYİ PARTİ": "İYİ PARTİ",
        "IYI PARTI": "İYİ PARTİ",
        "YENİ YOL": "YENİ YOL",
        "YENİ YOL PARTİSİ": "YENİ YOL",
        "SAADET": "SAADET PARTİSİ",
        "SAADET PARTİSİ": "SAADET PARTİSİ",
        "YENİDEN REFAH": "YENİDEN REFAH",
        "DEVA": "DEVA PARTİSİ",
        "DEVA PARTİSİ": "DEVA PARTİSİ",
        "DEMOKRAT PARTİ": "DEMOKRAT PARTİ",
        "DSP": "DSP",
        "TİP": "TİP",
        "DBP": "DBP",
        "HÜDA PAR": "HÜDA PAR",
        "EMEP": "EMEP",
        "BAĞIMSIZ": "BAĞIMSIZ",
    }

    return party_map.get(x_up, x_raw)


# ============================================================
# SCRAPE PAGE
# ============================================================

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(URL, headers=headers, timeout=60)
response.raise_for_status()

tables = pd.read_html(response.text)

candidate_tables = []

for table in tables:
    cols_text = " ".join(map(str, table.columns))
    body_text = " ".join(table.astype(str).head(5).values.ravel())

    if "Parti" in cols_text or "Parti" in body_text or "E-Posta" in cols_text or "E-Posta" in body_text:
        candidate_tables.append(table)

if not candidate_tables:
    raise ValueError("No suitable MP table found on the TBMM page.")

mv_df = candidate_tables[0].copy()

if isinstance(mv_df.columns, pd.MultiIndex):
    mv_df.columns = [" ".join(map(str, c)).strip() for c in mv_df.columns]

mv_df.columns = [norm_space(c) for c in mv_df.columns]

print("Raw columns:")
print(mv_df.columns.tolist())


# ============================================================
# FIND RELEVANT COLUMNS
# ============================================================

def find_col(possible_names):
    cols_lower = {c.lower(): c for c in mv_df.columns}

    for name in possible_names:
        if name.lower() in cols_lower:
            return cols_lower[name.lower()]

    for c in mv_df.columns:
        c_low = c.lower()
        for name in possible_names:
            if name.lower() in c_low:
                return c

    return None

name_col = find_col(["Adı-Soyadı", "Adı Soyadı", "Ad Soyad", "Adı", "Ad Soyadı"])
party_col = find_col(["Parti Adı", "Parti", "Siyasi Parti"])
city_col = find_col(["İl", "Il", "Şehir", "Sehir"])

if name_col is None or party_col is None or city_col is None:
    raise ValueError(
        f"Could not identify required columns. Found columns: {mv_df.columns.tolist()}"
    )


# ============================================================
# CLEAN LOOKUP TABLE
# ============================================================

mv_df = mv_df[[name_col, party_col, city_col]].copy()

mv_df = mv_df.rename(columns={
    name_col: "ad_soyad",
    party_col: "parti",
    city_col: "sehir",
})

mv_df["ad_soyad"] = mv_df["ad_soyad"].apply(norm_space)
mv_df["ad_soyad_clean"] = mv_df["ad_soyad"].apply(tr_upper)
mv_df["name_key"] = mv_df["ad_soyad"].apply(clean_key)

mv_df["parti"] = mv_df["parti"].apply(normalize_party)
mv_df["sehir"] = mv_df["sehir"].apply(norm_space)

mv_df = mv_df[
    mv_df["ad_soyad"].ne("")
    & mv_df["parti"].ne("")
    & mv_df["sehir"].ne("")
].copy()

mv_df = mv_df.drop_duplicates(subset=["name_key"], keep="first").reset_index(drop=True)

mv_df = mv_df[
    [
        "ad_soyad",
        "ad_soyad_clean",
        "name_key",
        "parti",
        "sehir",
    ]
]

print("Scraped MP rows:", len(mv_df))
display(mv_df.head(20))

print("\nParty counts:")
display(mv_df["parti"].value_counts(dropna=False))

print("\nCity counts:")
display(mv_df["sehir"].value_counts(dropna=False).head(20))


# ============================================================
# SAVE
# ============================================================

mv_df.to_csv(CSV_OUT, index=False, encoding="utf-8-sig")
mv_df.to_parquet(PARQUET_OUT, index=False)

print("Saved CSV to:", CSV_OUT)
print("Saved Parquet to:", PARQUET_OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_3449/1359532318.py:96: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


Raw columns:
['Adı-Soyadı', 'Parti Adı', 'İl', 'E-Posta']
Scraped MP rows: 591


,ad_soyad,ad_soyad_clean,name_key,parti,sehir
0,ABDULHAMİT GÜL,ABDULHAMİT GÜL,ABDULHAMİT GÜL,AK PARTİ,Gaziantep
1,ABDULKADİR AKGÜL,ABDULKADİR AKGÜL,ABDULKADİR AKGÜL,AK PARTİ,Yozgat
2,ABDULKADİR EMİN ÖNEN,ABDULKADİR EMİN ÖNEN,ABDULKADİR EMİN ÖNEN,AK PARTİ,Şanlıurfa
3,ABDULKADİR ÖZEL,ABDULKADİR ÖZEL,ABDULKADİR ÖZEL,AK PARTİ,Hatay
4,ABDULLAH AĞRALI,ABDULLAH AĞRALI,ABDULLAH AĞRALI,AK PARTİ,Konya
5,ABDULLAH DOĞRU,ABDULLAH DOĞRU,ABDULLAH DOĞRU,AK PARTİ,Adana
6,ABDULLAH GÜLER,ABDULLAH GÜLER,ABDULLAH GÜLER,AK PARTİ,Sivas
7,ABDURRAHİM FIRAT,ABDURRAHİM FIRAT,ABDURRAHİM FIRAT,AK PARTİ,Erzurum
8,ABDURRAHMAN BABACAN,ABDURRAHMAN BABACAN,ABDURRAHMAN BABACAN,AK PARTİ,Malatya
9,ABDURRAHMAN BAŞKAN,ABDURRAHMAN BAŞKAN,ABDURRAHMAN BAŞKAN,MHP,Antalya



Party counts:


,count
parti,
AK PARTİ,276
CHP,138
DEM PARTİ,56
MHP,46
İYİ PARTİ,29
YENİ YOL,20
BAĞIMSIZ,8
HÜDA PAR,4
YENİDEN REFAH,4



City counts:


,count
sehir,
İstanbul,96
Ankara,36
İzmir,28
Bursa,20
Antalya,17
Adana,15
Konya,15
Gaziantep,14
Şanlıurfa,14


Saved CSV to: /content/drive/MyDrive/tbmm_raw_tutanaklar/milletvekili_party_city_lookup.csv
Saved Parquet to: /content/drive/MyDrive/tbmm_raw_tutanaklar/milletvekili_party_city_lookup.parquet


In [ ]:
PARTY_LOOKUP_PATH = "/content/drive/MyDrive/tbmm_raw_tutanaklar/milletvekili_party_city_lookup.csv"

milletvekili_lookup = pd.read_csv(PARTY_LOOKUP_PATH)

display(milletvekili_lookup.head())
print(milletvekili_lookup.shape)

,ad_soyad,ad_soyad_clean,name_key,parti,sehir
0,ABDULHAMİT GÜL,ABDULHAMİT GÜL,ABDULHAMİT GÜL,AK PARTİ,Gaziantep
1,ABDULKADİR AKGÜL,ABDULKADİR AKGÜL,ABDULKADİR AKGÜL,AK PARTİ,Yozgat
2,ABDULKADİR EMİN ÖNEN,ABDULKADİR EMİN ÖNEN,ABDULKADİR EMİN ÖNEN,AK PARTİ,Şanlıurfa
3,ABDULKADİR ÖZEL,ABDULKADİR ÖZEL,ABDULKADİR ÖZEL,AK PARTİ,Hatay
4,ABDULLAH AĞRALI,ABDULLAH AĞRALI,ABDULLAH AĞRALI,AK PARTİ,Konya


(591, 5)


In [ ]:
import re
import pandas as pd

# ============================================================
# HELPERS
# ============================================================

TR_UPPER_TRANS = str.maketrans({
    "i": "İ",
    "ı": "I",
})

def norm_space(x):
    if pd.isna(x):
        return ""
    x = str(x).replace("\xa0", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def tr_upper(x):
    return norm_space(x).translate(TR_UPPER_TRANS).upper()

def clean_key(x):
    x = tr_upper(x)
    x = re.sub(r"[^\wÇĞİÖŞÜÂÎÛ ]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


# ============================================================
# PREPARE LOOKUP
# ============================================================

lookup = milletvekili_lookup.copy()

lookup["name_key"] = lookup["ad_soyad"].apply(clean_key)

lookup = lookup[["name_key", "parti", "sehir"]].drop_duplicates(
    subset=["name_key"],
    keep="first"
)


# ============================================================
# MERGE WITH SPEAKER DATA
# ============================================================

df_speaker_enriched["name_key"] = df_speaker_enriched["konusmaci_clean"].apply(clean_key)

df_speaker_enriched = df_speaker_enriched.merge(
    lookup,
    on="name_key",
    how="left",
    suffixes=("", "_lookup")
)


# ============================================================
# FILL PARTY IF MISSING
# ============================================================

party_missing_mask = (
    df_speaker_enriched["konusmaci_parti"].isna()
    & df_speaker_enriched["parti"].notna()
)

df_speaker_enriched.loc[party_missing_mask, "konusmaci_parti"] = (
    df_speaker_enriched.loc[party_missing_mask, "parti"]
)


# ============================================================
# FILL CITY IF MISSING
# Use TBMM city info when initial transcript city extraction is missing.
# This is mainly useful for rows like:
#   ALİ KARAOBA (Devamla)
# or names without city.
# ============================================================

NO_CITY_ROLES = {
    "başkan",
    "tbmm başkanı",
    "cumhurbaşkanı",
    "cumhurbaşkanı yardımcısı",
    "bakan",
    "devlet başkanı",
    "kamu başdenetçisi",
    "parse_error",
}

city_missing_mask = (
    df_speaker_enriched["konusmaci_sehir"].isna()
    & df_speaker_enriched["sehir"].notna()
    & ~df_speaker_enriched["konusmaci_tip"].isin(NO_CITY_ROLES)
)

df_speaker_enriched.loc[city_missing_mask, "konusmaci_sehir"] = (
    df_speaker_enriched.loc[city_missing_mask, "sehir"]
)


# ============================================================
# CLEAN MERGE HELPER COLUMNS
# ============================================================

df_speaker_enriched = df_speaker_enriched.drop(
    columns=["name_key", "parti", "sehir"],
    errors="ignore"
)


# ============================================================
# UPDATED ANALYSIS DATAFRAMES
# ============================================================

df_speaker_clean = df_speaker_enriched[
    ~df_speaker_enriched["speaker_parse_error"]
].copy()

df_speaker_parse_errors = df_speaker_enriched[
    df_speaker_enriched["speaker_parse_error"]
].copy()


# ============================================================
# CHECK RESULTS
# ============================================================

print("Rows:", len(df_speaker_enriched))
print("Missing party:", df_speaker_enriched["konusmaci_parti"].isna().sum())
print("Missing city:", df_speaker_enriched["konusmaci_sehir"].isna().sum())

display(
    df_speaker_enriched[
        [
            "konusmaci_raw",
            "konusmaci_clean",
            "konusmaci_tip",
            "konusmaci_parti",
            "konusmaci_sehir",
            "speaker_parse_error",
        ]
    ]
    .drop_duplicates()
    .head(80)
)

Rows: 199158
Missing party: 58921
Missing city: 74039


,konusmaci_raw,konusmaci_clean,konusmaci_tip,konusmaci_parti,konusmaci_sehir,speaker_parse_error
0,BAŞKAN DEVLET BAHÇELİ,DEVLET BAHÇELİ,başkan,MHP,<NA>,False
10,OSMAN GÖKÇEK (Ankara),OSMAN GÖKÇEK,milletvekili,AK PARTİ,Ankara,False
11,MURAT EMİR (Ankara),MURAT EMİR,milletvekili,CHP,Ankara,False
12,OSMAN GÖKÇEK (Devamla),OSMAN GÖKÇEK,milletvekili,AK PARTİ,Ankara,False
21,ŞAHİN TİN (Denizli),ŞAHİN TİN,milletvekili,AK PARTİ,Denizli,False
...,...,...,...,...,...,...
584,AYHAN BARUT (Adana),AYHAN BARUT,milletvekili,CHP,Adana,False
586,MUSTAFA ADIGÜZEL (Ordu),MUSTAFA ADIGÜZEL,milletvekili,CHP,Ordu,False
588,HASAN ÖZTÜRKMEN (Gaziantep),HASAN ÖZTÜRKMEN,milletvekili,CHP,Gaziantep,False
590,ORHAN SÜMER (Adana),ORHAN SÜMER,milletvekili,CHP,Adana,False


## Manual Patch for 28th Term MPs Whose Membership Ended

In [ ]:
import re
import os
import pandas as pd

# ============================================================
# MANUAL PATCH FOR 28TH TERM MPs WHOSE MEMBERSHIP ENDED
# Run this after creating/saving milletvekili_party_city_lookup.csv
# and before running your existing merge block.
# ============================================================

LOOKUP_PATH = "/content/drive/MyDrive/tbmm_raw_tutanaklar/milletvekili_party_city_lookup.csv"
LOOKUP_PARQUET_PATH = "/content/drive/MyDrive/tbmm_raw_tutanaklar/milletvekili_party_city_lookup.parquet"

TR_UPPER_TRANS = str.maketrans({
    "i": "İ",
    "ı": "I",
})

def norm_space(x):
    if pd.isna(x):
        return ""
    x = str(x).replace("\xa0", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def tr_upper(x):
    return norm_space(x).translate(TR_UPPER_TRANS).upper()

def clean_key(x):
    x = tr_upper(x)
    x = re.sub(r"[^\wÇĞİÖŞÜÂÎÛ ]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


# ============================================================
# 1. LOAD EXISTING LOOKUP
# ============================================================

milletvekili_lookup = pd.read_csv(LOOKUP_PATH)

# Make sure expected columns exist
for col in ["ad_soyad", "ad_soyad_clean", "name_key", "parti", "sehir"]:
    if col not in milletvekili_lookup.columns:
        milletvekili_lookup[col] = pd.NA


# ============================================================
# 2. MANUAL FORMER-MP ADDITIONS
# Source: TBMM "milletvekilliği sona erenler" detail pages
# ============================================================

manual_former_mps = pd.DataFrame([
    {
        "ad_soyad": "SIRRI SÜREYYA ÖNDER",
        "parti": "DEM PARTİ",
        "sehir": "İstanbul",
        "source": "former_manual",
    },
    {
        "ad_soyad": "MURAT KURUM",
        "parti": "AK PARTİ",
        "sehir": "İstanbul",
        "source": "former_manual",
    },
    {
        "ad_soyad": "HASAN BALTACI",
        "parti": "CHP",
        "sehir": "Kastamonu",
        "source": "former_manual",
    },
    {
        "ad_soyad": "ABDURRAHMAN TUTDERE",
        "parti": "CHP",
        "sehir": "Adıyaman",
        "source": "former_manual",
    },
    {
        "ad_soyad": "AHMET ÖNAL",
        "parti": "CHP",
        "sehir": "Kırıkkale",
        "source": "former_manual",
    },
    {
        "ad_soyad": "BURCU KÖKSAL",
        "parti": "CHP",
        "sehir": "Afyonkarahisar",
        "source": "former_manual",
    },
    {
        "ad_soyad": "ŞERAFETTİN CAN ATALAY",
        "parti": "TİP",
        "sehir": "Hatay",
        "source": "former_manual",
    },
    {
        "ad_soyad": "HASAN BİTMEZ",
        "parti": "SAADET PARTİSİ",
        "sehir": "Kocaeli",
        "source": "former_manual",
    },
])


# ============================================================
# 3. CREATE KEYS
# ============================================================

manual_former_mps["ad_soyad"] = manual_former_mps["ad_soyad"].apply(norm_space)
manual_former_mps["ad_soyad_clean"] = manual_former_mps["ad_soyad"].apply(tr_upper)
manual_former_mps["name_key"] = manual_former_mps["ad_soyad"].apply(clean_key)

milletvekili_lookup["ad_soyad"] = milletvekili_lookup["ad_soyad"].apply(norm_space)
milletvekili_lookup["ad_soyad_clean"] = milletvekili_lookup["ad_soyad"].apply(tr_upper)
milletvekili_lookup["name_key"] = milletvekili_lookup["ad_soyad"].apply(clean_key)

if "source" not in milletvekili_lookup.columns:
    milletvekili_lookup["source"] = "scraped"


# ============================================================
# 4. APPEND MANUAL ROWS
# Prefer existing scraped current rows if present,
# but use manual rows for missing names.
# ============================================================

combined_lookup = pd.concat(
    [
        milletvekili_lookup,
        manual_former_mps[
            ["ad_soyad", "ad_soyad_clean", "name_key", "parti", "sehir", "source"]
        ],
    ],
    ignore_index=True
)

source_priority = {
    "current": 0,
    "scraped": 1,
    "former": 2,
    "former_manual": 3,
}

combined_lookup["_source_priority"] = combined_lookup["source"].map(source_priority).fillna(9)

combined_lookup = (
    combined_lookup
    .sort_values(["name_key", "_source_priority"])
    .drop_duplicates(subset=["name_key"], keep="first")
    .drop(columns=["_source_priority"])
    .reset_index(drop=True)
)

combined_lookup = combined_lookup[
    ["ad_soyad", "ad_soyad_clean", "name_key", "parti", "sehir", "source"]
]


# ============================================================
# 5. SAVE PATCHED LOOKUP
# ============================================================

combined_lookup.to_csv(LOOKUP_PATH, index=False, encoding="utf-8-sig")
combined_lookup.to_parquet(LOOKUP_PARQUET_PATH, index=False)

print("Patched lookup saved to:", LOOKUP_PATH)
print("Rows after patch:", len(combined_lookup))

print("\nManual rows now in lookup:")
display(
    combined_lookup[
        combined_lookup["name_key"].isin(manual_former_mps["name_key"])
    ].sort_values("ad_soyad")
)


# ============================================================
# 6. OPTIONAL CHECK AGAINST YOUR SPEAKER DATAFRAME
# This only checks; your existing merge block can stay the same.
# ============================================================

if "df_speaker_enriched" in globals():
    temp_missing = (
        df_speaker_enriched[
            df_speaker_enriched["konusmaci_clean"].notna()
            & (
                df_speaker_enriched["konusmaci_parti"].isna()
                | df_speaker_enriched["konusmaci_sehir"].isna()
            )
        ][["konusmaci_raw", "konusmaci_clean", "konusmaci_parti", "konusmaci_sehir"]]
        .drop_duplicates()
        .copy()
    )

    temp_missing["name_key"] = temp_missing["konusmaci_clean"].apply(clean_key)

    matched_after_patch = temp_missing.merge(
        combined_lookup[["name_key", "parti", "sehir"]],
        on="name_key",
        how="left"
    )

    print("\nPreviously missing speakers that can now be filled from patched lookup:")
    display(
        matched_after_patch[
            matched_after_patch["parti"].notna()
            | matched_after_patch["sehir"].notna()
        ].head(100)
    )

Patched lookup saved to: /content/drive/MyDrive/tbmm_raw_tutanaklar/milletvekili_party_city_lookup.csv
Rows after patch: 599

Manual rows now in lookup:


,ad_soyad,ad_soyad_clean,name_key,parti,sehir,source
9,ABDURRAHMAN TUTDERE,ABDURRAHMAN TUTDERE,ABDURRAHMAN TUTDERE,CHP,Adıyaman,former_manual
35,AHMET ÖNAL,AHMET ÖNAL,AHMET ÖNAL,CHP,Kırıkkale,former_manual
86,BURCU KÖKSAL,BURCU KÖKSAL,BURCU KÖKSAL,CHP,Afyonkarahisar,former_manual
209,HASAN BALTACI,HASAN BALTACI,HASAN BALTACI,CHP,Kastamonu,former_manual
211,HASAN BİTMEZ,HASAN BİTMEZ,HASAN BİTMEZ,SAADET PARTİSİ,Kocaeli,former_manual
327,MURAT KURUM,MURAT KURUM,MURAT KURUM,AK PARTİ,İstanbul,former_manual
450,SIRRI SÜREYYA ÖNDER,SIRRI SÜREYYA ÖNDER,SIRRI SÜREYYA ÖNDER,DEM PARTİ,İstanbul,former_manual
596,ŞERAFETTİN CAN ATALAY,ŞERAFETTİN CAN ATALAY,ŞERAFETTİN CAN ATALAY,TİP,Hatay,former_manual



Previously missing speakers that can now be filled from patched lookup:


,konusmaci_raw,konusmaci_clean,konusmaci_parti,konusmaci_sehir,name_key,parti,sehir
0,BAŞKAN DEVLET BAHÇELİ,DEVLET BAHÇELİ,MHP,<NA>,DEVLET BAHÇELİ,MHP,Osmaniye
1,BURCU KÖKSAL (Afyonkarahisar),BURCU KÖKSAL,<NA>,Afyonkarahisar,BURCU KÖKSAL,CHP,Afyonkarahisar
2,HASAN BİTMEZ (Kocaeli),HASAN BİTMEZ,<NA>,Kocaeli,HASAN BİTMEZ,SAADET PARTİSİ,Kocaeli
3,HASAN BİTMEZ (Devamla),HASAN BİTMEZ,<NA>,Kocaeli,HASAN BİTMEZ,SAADET PARTİSİ,Kocaeli
5,BAŞKAN NUMAN KURTULMUŞ,NUMAN KURTULMUŞ,AK PARTİ,<NA>,NUMAN KURTULMUŞ,AK PARTİ,İstanbul
6,ABDURRAHMAN TUTDERE (Adıyaman),ABDURRAHMAN TUTDERE,<NA>,Adıyaman,ABDURRAHMAN TUTDERE,CHP,Adıyaman
8,HASAN BALTACI (Kastamonu),HASAN BALTACI,<NA>,Kastamonu,HASAN BALTACI,CHP,Kastamonu
9,BURCU KÖKSAL (Devamla),BURCU KÖKSAL,<NA>,Afyonkarahisar,BURCU KÖKSAL,CHP,Afyonkarahisar
12,ABDURRAHMAN TUTDERE (Devamla),ABDURRAHMAN TUTDERE,<NA>,Adıyaman,ABDURRAHMAN TUTDERE,CHP,Adıyaman
21,BAŞKAN SIRRI SÜREYYA ÖNDER,SIRRI SÜREYYA ÖNDER,<NA>,<NA>,SIRRI SÜREYYA ÖNDER,DEM PARTİ,İstanbul
